# 02 · Tüm revizyon deneyleri (K1–K5) — tek değerlendirme betiği

Makaledeki **bütün tablo ve şekiller** bu notebook'tan, aynı test kümeleri ve aynı eşiklerle üretilir (K4). Modeli eğitmez; 01 notebook'unda eğitilen modelleri kullanır.

| Bölüm | Madde | İçerik |
|---|---|---|
| A | — | Veri hazırlığı: Avenue (resmi etiketler otomatik iner), çapraz test klasörleri, UR Fall ADL + düşme (otomatik iner), Ped2 |
| B | K2 | **Kare adımı kontrolü:** `avenue_train` orijinal videolarla eşleştirilip eğitimde her kaçıncı karenin alındığı bulunur |
| C | K3, K2, K5 | Her model için: normal-veri eşiği, Avenue kare düzeyi AUC, çapraz test, UR Fall |
| D | **K1** | Kontrol deneyleri: (a) UR Fall alan içi ADL – düşme, (b) aynı video içinde düşme öncesi – düşme, (c) öğrenmesiz taban çizgileri |
| E | K2-b | UCSD Ped2 kare düzeyi AUC (`model_ped2.pth` varsa) |
| F | K5 | 3 seed özeti (ortalama ± std), video sayıları, video düzeyi bootstrap |
| G | K4 | Makale şekilleri + `makale_sayilari.md` (metne girecek bütün sayılar) |

**Çalıştırma:** T4 GPU → Tümünü çalıştır. İlk çalıştırma UR Fall indirmesi yüzünden uzun sürebilir (~20–40 dk). İndirilen her şey Drive'da önbelleğe alınır, sonraki çalıştırmalar hızlıdır.
Seed 7 / 2024 / Ped2 modelleri henüz yoksa ilgili kısımlar **atlanır**, notebook yine de çalışır. Modeller hazır olunca tekrar çalıştırın.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, json, glob, random, warnings, time, shutil, zipfile, tarfile, urllib.request
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn

from scipy.ndimage import gaussian_filter1d
from scipy.io import loadmat
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix, f1_score, precision_score

In [ ]:
class CFG:
    PROJ        = '/content/drive/MyDrive/AnomalyProject'
    BASE_DIR    = f'{PROJ}/frames'
    TRAIN_DIR   = f'{BASE_DIR}/avenue_train'
    AVENUE_RAW  = f'{PROJ}/data/Avenue_Dataset'
    AVENUE_GT   = f'{PROJ}/avenue_gt/testing_label_mask'
    AVENUE_GT_URL = 'https://www.cse.cuhk.edu.hk/leojia/projects/detectabnormal/ground_truth_demo.zip'
    # Avenue → UCF-Crime testi. Klasör adları eski; içerik dosya adlarından doğrulandı:
    #   ucf_robbery = Stealing019, ucf_accident = Burglary005 (kaza DEĞİL).
    # fall_test ÇIKARILDI: içinde 600 ADL (normal günlük aktivite) karesi "anomali" olarak etiketliydi.
    # Düşme testi artık kare etiketli UR Fall verisiyle yapılıyor (Avenue → UR Fall).
    CROSS = {'Normal': (f'{BASE_DIR}/ucf_normal', 0),
             'Kavga (Fighting)': (f'{BASE_DIR}/ucf_fighting', 1),
             'Hırsızlık (Stealing)': (f'{BASE_DIR}/ucf_robbery', 1),
             'Soygun (Burglary)': (f'{BASE_DIR}/ucf_accident', 1)}
    OLD_KEYS = {'Kavga (Fighting)': 'Kavga', 'Hırsızlık (Stealing)': 'Hırsızlık', 'Soygun (Burglary)': 'Kaza'}
    MODELS = {'seed42': f'{PROJ}/model_90plus.pth',          # makaledeki model
              'seed7': f'{PROJ}/model_seed7.pth',
              'seed2024': f'{PROJ}/model_seed2024.pth'}
    MAIN = 'seed42'                                         # şekiller ve ana tablo bu modelden
    PED2_MODEL = f'{PROJ}/model_ped2.pth'
    PED2_CACHE = f'{PROJ}/data/ped2_cache'
    PED2_URL   = 'http://www.svcl.ucsd.edu/projects/anomaly/UCSD_Anomaly_Dataset.tar.gz'
    URF_BASE   = 'http://fenix.ur.edu.pl/~mkepski/ds/data'
    URF_CACHE  = f'{PROJ}/data/urfall_cache'
    N_ADL, N_FALL = 40, 30
    LOG_DIR    = f'{PROJ}/egitim_kayitlari'
    CACHE_DIR  = f'{PROJ}/revizyon_sonuclari/cache'         # 1. turun önbelleği yeniden kullanılır
    OUT_DIR    = f'{PROJ}/revizyon_sonuclari_v2'
    TMP        = '/content/tmp'

    IMG_SIZE=64; SEQ_LEN=8; SEQ_STEP=2; BATCH=64; VAL_FRAC=0.10; VAL_SPLIT_SEED=42
    AVENUE_STRIDE_DEFAULT = 2   # B bölümü otomatik bulamazsa kullanılır
    URF_STRIDE   = 1            # UR Fall pencere içi kare aralığı (makaledeki fall_test ile aynı: ardışık kare)
    PED2_STRIDE  = 1
    N_BOOT=1000; BOOT_BLOCK=16; N_BALANCED=10; SEED=42
    MAIN_THR = 'P95'            # ana eşik önceden sabitlendi; test sonucuna göre değiştirmeyin

for d in [CFG.OUT_DIR, CFG.CACHE_DIR, CFG.TMP, CFG.URF_CACHE, f'{CFG.OUT_DIR}/sekiller', f'{CFG.OUT_DIR}/tablolar', f'{CFG.OUT_DIR}/skorlar']:
    os.makedirs(d, exist_ok=True)
random.seed(CFG.SEED); np.random.seed(CFG.SEED); torch.manual_seed(CFG.SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Cihaz:', DEVICE)
print('Bulunan modeller:', {k: os.path.exists(v) for k, v in CFG.MODELS.items()}, '| Ped2:', os.path.exists(CFG.PED2_MODEL))

In [ ]:
EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

def natural_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', str(s))]

def list_frames(folder, natural=True):
    fs = [f for f in os.listdir(folder) if f.lower().endswith(EXTS)]
    fs = sorted(fs, key=natural_key) if natural else sorted(fs)
    return [os.path.join(folder, f) for f in fs]

def to_gray64(img):
    # Ana notebook'taki load_frame ile aynı: gri ton + 64x64 BILINEAR, [0,1]
    if not isinstance(img, Image.Image): img = Image.fromarray(img)
    return np.asarray(img.convert('L').resize((CFG.IMG_SIZE, CFG.IMG_SIZE), Image.BILINEAR),
                      np.float32) / 255.0

def load_stack(paths, cache_name=None, cache_dir=None):
    cache_dir = cache_dir or CFG.CACHE_DIR
    if cache_name:
        cp = os.path.join(cache_dir, cache_name + '.npy')
        if os.path.exists(cp):
            arr = np.load(cp)
            if len(arr) == len(paths): return arr
    arr = np.stack([to_gray64(Image.open(p)) for p in paths])
    if cache_name:
        os.makedirs(cache_dir, exist_ok=True); np.save(os.path.join(cache_dir, cache_name + '.npy'), arr)
    return arr

def cached_array(name, fn, cache_dir=None):
    cp = os.path.join(cache_dir or CFG.CACHE_DIR, name + '.npy')
    if os.path.exists(cp): return np.load(cp)
    arr = fn(); os.makedirs(os.path.dirname(cp), exist_ok=True); np.save(cp, arr); return arr

def frames_from_avi(path):
    import cv2
    cap, out = cv2.VideoCapture(path), []
    while True:
        ok, f = cap.read()
        if not ok: break
        out.append(to_gray64(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)))
    cap.release()
    return np.stack(out)

def fetch(url, dst, tries=3):
    # Dosyayı indirir; zaten varsa atlar. Başarısızsa False döner.
    if os.path.exists(dst) and os.path.getsize(dst) > 0: return True
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for k in range(tries):
        try:
            urllib.request.urlretrieve(url, dst + '.part'); os.replace(dst + '.part', dst); return True
        except Exception as e:
            print(f'   indirme hatası ({k+1}/{tries}) {os.path.basename(url)}: {e}')
            time.sleep(3)
    return False

# ── UCSD Ped2 ─────────────────────────────────────────────────────────
def parse_ped_gt(m_path):
    txt = open(m_path, encoding='latin-1').read()
    out = []
    for body in re.findall(r'gt_frame\s*=\s*\[([^\]]*)\]', txt):
        fr = []
        for a, b in re.findall(r'(\d+)\s*:\s*(\d+)', body): fr += list(range(int(a), int(b) + 1))
        rest = re.sub(r'\d+\s*:\s*\d+', ' ', body)
        fr += [int(x) for x in re.findall(r'\d+', rest)]
        out.append(sorted(set(fr)))
    return out

def prepare_ped2():
    cache = CFG.PED2_CACHE
    if os.path.exists(f'{cache}/done.json'):
        meta = json.load(open(f'{cache}/done.json'))
        tr = [np.load(f'{cache}/train_{i:02d}.npy') for i in range(meta['n_train'])]
        te = [np.load(f'{cache}/test_{i:02d}.npy') for i in range(meta['n_test'])]
        lb = [np.load(f'{cache}/label_{i:02d}.npy') for i in range(meta['n_test'])]
        return tr, te, lb
    tgz = f'{CFG.TMP}/UCSD_Anomaly_Dataset.tar.gz'
    print('UCSD veri seti indiriliyor (~700 MB)...')
    if not fetch(CFG.PED2_URL, tgz): raise RuntimeError('Ped2 indirilemedi: ' + CFG.PED2_URL)
    ex = f'{CFG.TMP}/ucsd'
    if not os.path.exists(ex):
        with tarfile.open(tgz) as t: t.extractall(ex)
    root = next(r for r, ds, _ in os.walk(ex) if os.path.basename(r) == 'UCSDped2')
    trd = sorted([d for d in glob.glob(f'{root}/Train/*') if re.fullmatch(r'Train\d+', os.path.basename(d))], key=natural_key)
    ted = sorted([d for d in glob.glob(f'{root}/Test/*') if re.fullmatch(r'Test\d+', os.path.basename(d))], key=natural_key)
    gt  = parse_ped_gt(glob.glob(f'{root}/Test/*.m')[0])
    os.makedirs(cache, exist_ok=True)
    tr, te, lb = [], [], []
    for i, d in enumerate(trd):
        a = load_stack(list_frames(d)); np.save(f'{cache}/train_{i:02d}.npy', a); tr.append(a)
    for i, d in enumerate(ted):
        a = load_stack(list_frames(d)); l = np.zeros(len(a), int)
        for f in gt[i]:
            if 1 <= f <= len(a): l[f - 1] = 1
        np.save(f'{cache}/test_{i:02d}.npy', a); np.save(f'{cache}/label_{i:02d}.npy', l); te.append(a); lb.append(l)
    json.dump({'n_train': len(tr), 'n_test': len(te)}, open(f'{cache}/done.json', 'w'))
    print(f'Ped2 hazır: {len(tr)} eğitim, {len(te)} test videosu')
    return tr, te, lb

In [ ]:
class CB3(nn.Module):
    def __init__(self, ic, oc, k=(3,3,3), s=1, p=1):
        super().__init__()
        self.net = nn.Sequential(nn.Conv3d(ic, oc, k, s, p, bias=False),
                                 nn.BatchNorm3d(oc), nn.LeakyReLU(0.2, inplace=True))
    def forward(self, x): return self.net(x)

class AE3D(nn.Module):
    # skip = (E1, E2, E3) seviyelerinde atlama bağlantısı var mı. Varsayılan: hepsi (makaledeki model).
    def __init__(self, skip=(True, True, True)):
        super().__init__()
        self.skip = tuple(skip); s1, s2, s3 = self.skip
        self.e1=nn.Sequential(CB3(1,32),CB3(32,32));     self.p1=nn.MaxPool3d((1,2,2),(1,2,2))
        self.e2=nn.Sequential(CB3(32,64),CB3(64,64));    self.p2=nn.MaxPool3d(2,2)
        self.e3=nn.Sequential(CB3(64,128),CB3(128,128)); self.p3=nn.MaxPool3d(2,2)
        self.bn=nn.Sequential(CB3(128,128),CB3(128,64),CB3(64,128),nn.Dropout3d(0.4))
        self.u3=nn.Upsample(scale_factor=(2,2,2),mode='trilinear',align_corners=False)
        self.d3=nn.Sequential(CB3(256 if s3 else 128,128),CB3(128,64))
        self.u2=nn.Upsample(scale_factor=(2,2,2),mode='trilinear',align_corners=False)
        self.d2=nn.Sequential(CB3(128 if s2 else 64,64),CB3(64,32))
        self.u1=nn.Upsample(scale_factor=(1,2,2),mode='trilinear',align_corners=False)
        self.d1=nn.Sequential(CB3(64 if s1 else 32,32),CB3(32,32))
        self.out=nn.Sequential(nn.Conv3d(32,1,1),nn.Sigmoid())
        for m in self.modules():
            if isinstance(m, nn.Conv3d): nn.init.kaiming_normal_(m.weight, nonlinearity='leaky_relu')
            elif isinstance(m, nn.BatchNorm3d): nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
    def forward(self,x):
        e1=self.e1(x); e1p=self.p1(e1)
        e2=self.e2(e1p); e2p=self.p2(e2)
        e3=self.e3(e2p); e3p=self.p3(e3)
        b=self.bn(e3p)
        s1, s2, s3 = self.skip
        d=self.u3(b);  d=self.d3(torch.cat([d,e3],1) if s3 else d)
        d=self.u2(d);  d=self.d2(torch.cat([d,e2],1) if s2 else d)
        d=self.u1(d);  d=self.d1(torch.cat([d,e1],1) if s1 else d)
        return self.out(d)

def win_idx(n, step, stride=1):
    # 8 karelik pencereler: başlangıç adımı `step`, pencere içi kare aralığı `stride`
    span = (CFG.SEQ_LEN - 1) * stride + 1
    return [[s + t * stride for t in range(CFG.SEQ_LEN)] for s in range(0, n - span + 1, step)]

def load_model(path):
    m = AE3D().to(DEVICE); m.load_state_dict(torch.load(path, map_location=DEVICE)); m.eval(); return m

@torch.no_grad()
def score_windows(model, frames, idx, return_recon=False):
    out, recs = [], []
    ft = torch.from_numpy(np.ascontiguousarray(frames))
    for i in range(0, len(idx), CFG.BATCH):
        b = torch.as_tensor(np.asarray(idx[i:i+CFG.BATCH]))
        x = ft[b].unsqueeze(1).to(DEVICE); r = model(x)
        out.append(((r - x) ** 2).mean(dim=(1,2,3,4)).cpu().numpy())
        if return_recon: recs.append(r[:, 0].cpu().numpy())
    s = np.concatenate(out) if out else np.zeros(0)
    return (s, np.concatenate(recs)) if return_recon else s

def frame_level(frames, stride, window_scorer):
    # Her kareye, onu içeren 8'li pencerelerin ortalama skoru (pencere adımı 1)
    n = len(frames); span = (CFG.SEQ_LEN - 1) * stride + 1
    if n < span: frames = np.concatenate([frames, np.repeat(frames[-1:], span - n, 0)])
    idx = win_idx(len(frames), 1, stride); ws = window_scorer(frames, idx)
    acc = np.zeros(len(frames)); cnt = np.zeros(len(frames))
    for w, s in zip(idx, ws): acc[w] += s; cnt[w] += 1
    return (acc / np.maximum(cnt, 1))[:n]

def auc(y, s):
    y = np.asarray(y); return float(roc_auc_score(y, s)) if 0 < y.mean() < 1 else float('nan')
def minmax(x): return (x - x.min()) / (x.max() - x.min() + 1e-12)
pct = lambda x: round(100 * float(x), 2)

## A · Veri hazırlığı

In [ ]:
# A1 · Avenue resmi etiketleri (yoksa indir)
if not os.path.exists(CFG.AVENUE_GT):
    z = f'{CFG.TMP}/gt.zip'
    assert fetch(CFG.AVENUE_GT_URL, z), 'Avenue etiketleri indirilemedi'
    with zipfile.ZipFile(z) as f: f.extractall(f'{CFG.TMP}/gt')
    src = next(r for r, ds, _ in os.walk(f'{CFG.TMP}/gt') if os.path.basename(r) == 'testing_label_mask')
    shutil.copytree(src, CFG.AVENUE_GT)
mats = sorted(glob.glob(f'{CFG.AVENUE_GT}/*_label.mat'), key=natural_key)
av_labels = []
for m in mats:
    vol = loadmat(m)['volLabel']
    av_labels.append(np.array([int(np.asarray(vol[0, i]).sum() > 0) for i in range(vol.shape[1])]))

# A2 · Avenue test kareleri: orijinal videolardan (video sınırları kesin)
avis = sorted(glob.glob(f'{CFG.AVENUE_RAW}/testing_videos/*.avi'), key=natural_key)
assert len(avis) == 21, f'testing_videos içinde 21 video bekleniyordu, {len(avis)} bulundu'
av_frames = [cached_array(f'avenue_avi_{i+1:02d}', lambda p=p: frames_from_avi(p)) for i, p in enumerate(avis)]
for i, (a, l) in enumerate(zip(av_frames, av_labels)):
    assert len(a) == len(l), f'Video {i+1}: {len(a)} kare, {len(l)} etiket'
print(f'Avenue test: {len(av_frames)} video, {sum(map(len, av_frames))} kare, anomali oranı %{100*np.mean(np.concatenate(av_labels)):.1f}')

# A3 · Avenue eğitim kareleri (eğitimdeki sırayla) ve doğrulama pencereleri
train_paths  = list_frames(CFG.TRAIN_DIR, natural=False)
train_frames = load_stack(train_paths, 'avenue_train')
tr_starts = win_idx(len(train_frames), CFG.SEQ_STEP, 1)
n_seq = len(tr_starts); val_n = max(1, int(n_seq * CFG.VAL_FRAC))
_, val_sub = torch.utils.data.random_split(range(n_seq), [n_seq - val_n, val_n],
                                           generator=torch.Generator().manual_seed(CFG.VAL_SPLIT_SEED))
val_idx = [tr_starts[i] for i in val_sub.indices]
print(f'Avenue eğitim: {len(train_frames)} kare → {n_seq} sekans, doğrulama {len(val_idx)}')

# A4 · Çapraz test (makaledeki kurulum, tüm sekanslar)
cross = {}
for name, (folder, lbl) in CFG.CROSS.items():
    paths = list_frames(folder, natural=False)
    arr = load_stack(paths, 'cross_' + os.path.basename(folder))
    cross[name] = dict(frames=arr, idx=win_idx(len(arr), CFG.SEQ_STEP, 1), label=lbl,
                       names=[os.path.basename(p) for p in paths])
    print(f'  {name:<10} {len(arr):>5} kare → {len(cross[name]["idx"]):>4} sekans')

In [ ]:
# A5 · UR Fall: 40 ADL (normal günlük aktivite) + 30 düşme videosu, kamera 0, RGB
def urf_sequence(name):
    cp, cn = f'{CFG.URF_CACHE}/{name}.npy', f'{CFG.URF_CACHE}/{name}_no.npy'
    if os.path.exists(cp): return np.load(cp), np.load(cn)
    z = f'{CFG.TMP}/{name}-cam0-rgb.zip'
    if not fetch(f'{CFG.URF_BASE}/{name}-cam0-rgb.zip', z): return None, None
    ex = f'{CFG.TMP}/{name}'
    with zipfile.ZipFile(z) as f: f.extractall(ex)
    pngs = sorted(glob.glob(f'{ex}/**/*.png', recursive=True), key=natural_key)
    arr = load_stack(pngs)
    no  = np.array([int(re.findall(r'(\d+)', os.path.basename(p))[-1]) for p in pngs])
    np.save(cp, arr); np.save(cn, no); shutil.rmtree(ex, ignore_errors=True); os.remove(z)
    return arr, no

urf = {}; fails = 0
t0 = time.time()
for kind, n in [('adl', CFG.N_ADL), ('fall', CFG.N_FALL)]:
    for i in range(1, n + 1):
        if fails >= 2 and not urf: break                      # sunucuya hiç ulaşılamıyor → UR Fall atlanır
        a, no = urf_sequence(f'{kind}-{i:02d}')
        if a is not None: urf[f'{kind}-{i:02d}'] = dict(frames=a, no=no, kind=kind)
        else: fails += 1
if not urf: print('⚠️ UR Fall indirilemedi; K1-a/K1-b atlanacak. Zip dosyalarını elle indirip CFG.TMP içine koyabilirsiniz.')
print(f"UR Fall: {sum(v['kind']=='adl' for v in urf.values())} ADL, {sum(v['kind']=='fall' for v in urf.values())} düşme videosu ({time.time()-t0:.0f} sn)")

# Düşme videolarının kare etiketleri: -1 = düşme öncesi, 0 = düşüyor, 1 = yerde
lab_csv = f'{CFG.URF_CACHE}/urfall-cam0-falls.csv'
if urf and not fetch(f'{CFG.URF_BASE}/urfall-cam0-falls.csv', lab_csv):
    urf = {k: v for k, v in urf.items() if v['kind'] == 'adl'}; print('⚠️ Düşme etiket dosyası indirilemedi')
fl = (pd.read_csv(lab_csv, header=None).iloc[:, :3] if os.path.exists(lab_csv) else pd.DataFrame(columns=[0, 1, 2])); fl.columns = ['seq', 'frame', 'label']
for k, v in urf.items():
    if v['kind'] == 'adl': v['flab'] = np.full(len(v['frames']), -1)
    else:
        d = dict(zip(fl[fl.seq == k].frame, fl[fl.seq == k].label))
        v['flab'] = np.array([d.get(int(x), -9) for x in v['no']])    # -9: etiketsiz kare

In [ ]:
# A6 · UR Fall pencereleri ve iki test kurulumu
#   K1-a  Alan içi:      ADL penceresi (normal)  vs  düşme/yerde penceresi (anomali)
#   K1-b  Aynı video içi: düşme videosunun olay öncesi penceresi  vs  aynı videonun olay penceresi
for k, v in urf.items():
    v['idx'] = win_idx(len(v['frames']), CFG.SEQ_STEP, CFG.URF_STRIDE)
    wl = np.array([v['flab'][w] for w in v['idx']]) if len(v['idx']) else np.zeros((0, CFG.SEQ_LEN))
    v['valid'] = (wl != -9).all(1) if len(wl) else np.zeros(0, bool)
    v['event'] = (wl >= 0).any(1) if len(wl) else np.zeros(0, bool)

def urf_sets(scores):
    # scores: {video: pencere skorları} → iki kurulumun (y, s) dizileri
    a_y, a_s, b_y, b_s = [], [], [], []
    for k, v in urf.items():
        s, ok = scores[k], urf[k]['valid']
        if v['kind'] == 'adl':
            a_y += [0] * int(ok.sum()); a_s += list(s[ok])
        else:
            ev = v['event'] & ok; pre = (~v['event']) & ok
            a_y += [1] * int(ev.sum()); a_s += list(s[ev])
            b_y += [0] * int(pre.sum()) + [1] * int(ev.sum()); b_s += list(s[pre]) + list(s[ev])
    return (np.array(a_y), np.array(a_s)), (np.array(b_y), np.array(b_s))

if urf:
    (ya, _), (yb, _) = urf_sets({k: np.zeros(len(v['idx'])) for k, v in urf.items()})
    print(f'K1-a: {np.sum(ya==0)} ADL, {np.sum(ya==1)} düşme penceresi | K1-b: {np.sum(yb==0)} olay öncesi, {np.sum(yb==1)} olay penceresi')

## B · Kare adımı kontrolü (Avenue sonucu buna bağlı)
`avenue_train` içindeki 7.658 kare orijinal 16 eğitim videosunun kareleriyle eşleştirilir. Ardışık iki eğitim karesi videoda kaç kare arayla duruyorsa, test pencereleri de aynı aralıkla kurulur.

In [ ]:
tr_avis = sorted(glob.glob(f'{CFG.AVENUE_RAW}/training_videos/*.avi'), key=natural_key)
AV_STRIDE, stride_note = CFG.AVENUE_STRIDE_DEFAULT, 'varsayılan (eğitim videoları bulunamadı)'
if tr_avis:
    tv = [cached_array(f'avenue_train_avi_{i+1:02d}', lambda p=p: frames_from_avi(p)) for i, p in enumerate(tr_avis)]
    small = lambda a: a.reshape(len(a), 16, 4, 16, 4).mean((2, 4)).reshape(len(a), -1)
    bank  = np.concatenate([small(v) for v in tv]); vid = np.concatenate([[i]*len(v) for i, v in enumerate(tv)])
    pos   = np.concatenate([np.arange(len(v)) for v in tv])
    probe = small(train_frames[:400])
    d = ((probe**2).sum(1)[:, None] + (bank**2).sum(1)[None] - 2 * probe @ bank.T) / probe.shape[1]   # bellek dostu MSE
    j = d.argmin(1); err = d.min(1)
    same = vid[j][1:] == vid[j][:-1]; steps = (pos[j][1:] - pos[j][:-1])[same]
    steps = steps[steps > 0]
    if len(steps) and np.median(err) < 1e-3:
        AV_STRIDE = int(np.median(steps))
        stride_note = f'eşleştirme: medyan adım {AV_STRIDE}, eşleşme hatası medyanı {np.median(err):.1e}, adımların %{100*np.mean(steps==AV_STRIDE):.0f}\'ı {AV_STRIDE}'
    else:
        stride_note = f'eşleştirme güvenilir değil (hata {np.median(err):.1e}), varsayılan kullanıldı'
    print(f'Eğitim videosu kare sayısı toplamı: {sum(map(len, tv))} | avenue_train: {len(train_frames)}')
print(f'➡  Avenue test pencere içi kare aralığı = {AV_STRIDE}  ({stride_note})')

## C · Her model için değerlendirme (K2, K3, çapraz test, UR Fall)
Her model için skorlar `skorlar/` klasörüne kaydedilir. Tekrar çalıştırınca yeniden hesaplanmaz.

In [ ]:
def migrate(d):
    # Önceki çalıştırmadaki eski kategori adlarını yenilerine taşır (aynı klasör → aynı skor)
    for new, old in CFG.OLD_KEYS.items():
        if new not in d and old in d: d[new] = d.pop(old)
    for k in [k for k in d if k not in CFG.CROSS]: d.pop(k)

def thresholds_from(val_scores):
    mu, sd = val_scores.mean(), val_scores.std()
    return {'P95': float(np.percentile(val_scores, 95)), 'P99': float(np.percentile(val_scores, 99)),
            'Ort+2σ': float(mu + 2*sd), 'Ort+3σ': float(mu + 3*sd)}

def evaluate_model(tag, path):
    sp = f'{CFG.OUT_DIR}/skorlar/{tag}.npz'
    if os.path.exists(sp):
        z = np.load(sp, allow_pickle=True); S = z['S'].item(); print(f'{tag}: kayıtlı skorlar yüklendi')
        migrate(S['cross']); return S
    model = load_model(path); t0 = time.time(); S = {}
    scorer = lambda fr, idx: score_windows(model, fr, idx)
    S['val'] = scorer(train_frames, val_idx)
    strides = sorted({AV_STRIDE, 1, 2}) if tag == CFG.MAIN else [AV_STRIDE]
    S['avenue'] = {st: [frame_level(f, st, scorer) for f in av_frames] for st in strides}
    S['cross'] = {k: scorer(v['frames'], v['idx']) for k, v in cross.items()}
    S['urf'] = {k: (scorer(v['frames'], v['idx']) if len(v['idx']) else np.zeros(0)) for k, v in urf.items()}
    np.savez(sp, S=np.array(S, dtype=object)); print(f'{tag}: skorlandı ({time.time()-t0:.0f} sn)')
    return S

def summarize(S):
    R = {}; thr = thresholds_from(S['val']); R['thr'] = thr
    y = np.concatenate(av_labels)
    for st, sc in S['avenue'].items():
        R[f'av_micro_s{st}'] = auc(y, np.concatenate(sc))
        R[f'av_macro_s{st}'] = float(np.nanmean([auc(l, s) for l, s in zip(av_labels, sc)]))
        R[f'av_micro_mm_s{st}'] = auc(y, np.concatenate([minmax(s) for s in sc]))
    sa = np.concatenate(S['avenue'][AV_STRIDE])
    for k, t in thr.items():
        p = sa >= t; R[f'av_tpr_{k}'] = p[y==1].mean(); R[f'av_fpr_{k}'] = p[y==0].mean()
    cats = [k for k in cross if cross[k]['label'] == 1]
    ys = np.concatenate([np.full(len(S['cross'][k]), cross[k]['label']) for k in cross])
    ss = np.concatenate([S['cross'][k] for k in cross])
    R['cr_auc'] = auc(ys, ss); R['cr_ap'] = float(average_precision_score(ys, ss))
    n = S['cross']['Normal']
    for c in cats:
        R[f'cr_auc_{c}'] = auc(np.r_[np.zeros(len(n)), np.ones(len(S['cross'][c]))], np.r_[n, S['cross'][c]])
    for k, t in thr.items():
        R[f'cr_fpr_{k}'] = float((n >= t).mean()); R[f'cr_rec_{k}'] = float((ss[ys==1] >= t).mean())
        for c in cats: R[f'cr_det_{c}_{k}'] = float((S['cross'][c] >= t).mean())
    if urf:
        (ya, sa_), (yb, sb) = urf_sets(S['urf'])
        R['urf_a'] = auc(ya, sa_); R['urf_b'] = auc(yb, sb)
        for k, t in thr.items():
            R[f'urf_fpr_{k}'] = float((sa_[ya==0] >= t).mean()); R[f'urf_rec_{k}'] = float((sa_[ya==1] >= t).mean())
    return R

SCORES, RES = {}, {}
for tag, path in CFG.MODELS.items():
    if not os.path.exists(path): print(f'{tag}: model yok, atlandı'); continue
    SCORES[tag] = evaluate_model(tag, path); RES[tag] = summarize(SCORES[tag])
M, R0 = SCORES[CFG.MAIN], RES[CFG.MAIN]
print(f"\nAna model ({CFG.MAIN}) | Avenue mikro AUC (adım {AV_STRIDE}): %{100*R0[f'av_micro_s{AV_STRIDE}']:.2f} | UCF-Crime AUC: %{100*R0['cr_auc']:.2f}"
      + (f" | UR Fall ADL–düşme AUC: %{100*R0['urf_a']:.2f}" if urf else ''))

In [ ]:
# C1 · Ana model tabloları
T = {}
T['avenue'] = pd.DataFrame([{'Pencere içi kare aralığı': f'{st}' + (' (eğitimle uyumlu)' if st == AV_STRIDE else ''),
                             'Mikro AUC (%)': pct(R0[f'av_micro_s{st}']), 'Makro AUC (%)': pct(R0[f'av_macro_s{st}']),
                             'Mikro AUC, video bazında min-max (%)': pct(R0[f'av_micro_mm_s{st}'])}
                            for st in M['avenue']])
T['esikler'] = pd.DataFrame([{'Eşik': k, 'Değer': f'{v:.2e}',
    'Avenue TPR (%)': pct(R0[f'av_tpr_{k}']), 'Avenue FPR (%)': pct(R0[f'av_fpr_{k}']),
    'UCF-Crime: yanlış alarm (%)': pct(R0[f'cr_fpr_{k}']), 'UCF-Crime: recall (%)': pct(R0[f'cr_rec_{k}']),
    **({'UR Fall: yanlış alarm ADL (%)': pct(R0[f'urf_fpr_{k}']), 'UR Fall: recall (%)': pct(R0[f'urf_rec_{k}'])} if urf else {})}
    for k, v in R0['thr'].items()])
cats = [k for k in cross if cross[k]['label'] == 1]
T['kategori'] = pd.DataFrame([{'Kategori': c, 'ROC-AUC (%)': pct(R0[f'cr_auc_{c}']),
    'Tespit P95 (%)': pct(R0[f'cr_det_{c}_P95']), 'Tespit P99 (%)': pct(R0[f'cr_det_{c}_P99']),
    'Ort. MSE / normal ort. MSE': round(float(M['cross'][c].mean() / M['cross']['Normal'].mean()), 2)} for c in cats])
for k in ['avenue', 'esikler', 'kategori']: display(T[k])

## D · K1 — Kontrol deneyleri
Aynı test kurulumlarında model ile **öğrenmesiz** skorlar karşılaştırılır:
- **Kare farkı enerjisi:** ardışık kareler arası ortalama mutlak fark (hareket miktarı)
- **Optik akış büyüklüğü:** Farnebäck optik akışının ortalama büyüklüğü
- **Parlaklık sapması:** penceredeki ortalama parlaklığın Avenue eğitim ortalamasından farkı
- **Kontrast sapması:** kare standart sapmasının Avenue eğitim değerinden farkı
- **Avenue ortalama kareye uzaklık:** pencerenin, Avenue eğitim karelerinin ortalamasına (sabit arka plan) MSE uzaklığı

Son üçü olayla hiç ilgilenmez, sadece "görüntü Avenue'ya benziyor mu" sorusunu ölçer. Bu skorlar çapraz testte yüksek AUC verip Avenue ve UR Fall içinde düşük kalıyorsa, çapraz testteki başarının bir kısmı alan farkından geliyor demektir.

In [ ]:
import cv2
MU_TR, SD_TR = float(train_frames.mean()), float(train_frames.std(axis=(1, 2)).mean())
AVG_TR = train_frames.mean(0)

def _flow_mag(a, b):
    f = cv2.calcOpticalFlowFarneback((a*255).astype(np.uint8), (b*255).astype(np.uint8), None, 0.5, 2, 9, 3, 5, 1.1, 0)
    return float(np.sqrt((f ** 2).sum(-1)).mean())

def baseline_scorer(kind):
    def fn(frames, idx):
        out = []
        for i in range(0, len(idx), 512):
            W = frames[np.asarray(idx[i:i+512])]                      # [B,T,H,W]
            if kind == 'diff':     out.append(np.abs(np.diff(W, axis=1)).mean((1, 2, 3)))
            elif kind == 'bright': out.append(np.abs(W.mean((1, 2, 3)) - MU_TR))
            elif kind == 'contr':  out.append(np.abs(W.std((2, 3)).mean(1) - SD_TR))
            elif kind == 'avgdist':out.append(((W - AVG_TR) ** 2).mean((1, 2, 3)))
            elif kind == 'flow':
                cache = {}
                def fm(a, b):
                    if (a, b) not in cache: cache[(a, b)] = _flow_mag(frames[a], frames[b])
                    return cache[(a, b)]
                out.append(np.array([np.mean([fm(w[t], w[t+1]) for t in range(len(w)-1)]) for w in idx[i:i+512]]))
        return np.concatenate(out) if out else np.zeros(0)
    return fn

BASELINES = {'Kare farkı enerjisi': 'diff', 'Optik akış büyüklüğü': 'flow', 'Parlaklık sapması': 'bright',
             'Kontrast sapması': 'contr', 'Avenue ortalama kareye uzaklık': 'avgdist'}
bp = f'{CFG.OUT_DIR}/skorlar/baselines.npz'
if os.path.exists(bp):
    BS = np.load(bp, allow_pickle=True)['B'].item()
    for v in BS.values(): migrate(v['cross'])
else:
    BS = {}; t0 = time.time()
    for name, kind in BASELINES.items():
        f = baseline_scorer(kind)
        BS[name] = {'avenue': [frame_level(a, AV_STRIDE, f) for a in av_frames],
                    'cross': {k: f(v['frames'], v['idx']) for k, v in cross.items()},
                    'urf': {k: (f(v['frames'], v['idx']) if len(v['idx']) else np.zeros(0)) for k, v in urf.items()}}
        print(f'  {name}: {time.time()-t0:.0f} sn')
    np.savez(bp, B=np.array(BS, dtype=object))

def k1_row(name, S):
    y = np.concatenate(av_labels)
    ys = np.concatenate([np.full(len(S['cross'][k]), cross[k]['label']) for k in cross])
    ss = np.concatenate([S['cross'][k] for k in cross])
    av = S['avenue'][AV_STRIDE] if isinstance(S['avenue'], dict) else S['avenue']
    r = {'Yöntem': name, 'Avenue (alan içi, kare) AUC %': pct(auc(y, np.concatenate(av))),
         'Avenue→UCF-Crime AUC %': pct(auc(ys, ss))}
    if urf:
        (ya, sa_), (yb, sb) = urf_sets(S['urf'])
        r['Avenue→UR Fall ADL–düşme AUC % (K1-a)'] = pct(auc(ya, sa_))
        r['UR Fall aynı video içi AUC % (K1-b)'] = pct(auc(yb, sb))
    return r

T['k1'] = pd.DataFrame([k1_row(f'3B otokodlayıcı ({CFG.MAIN})', M)] + [k1_row(n, s) for n, s in BS.items()])
display(T['k1'])

## E · K2-b — UCSD Ped2 (ayrı eğitilmiş model, aynı mimari ve protokol)

In [ ]:
PED2 = None
if os.path.exists(CFG.PED2_MODEL):
    _, p_te, p_lb = prepare_ped2()
    pm = load_model(CFG.PED2_MODEL); sc = lambda fr, idx: score_windows(pm, fr, idx)
    p_sc = [frame_level(a, CFG.PED2_STRIDE, sc) for a in p_te]
    yp = np.concatenate(p_lb)
    PED2 = {'micro': auc(yp, np.concatenate(p_sc)), 'macro': float(np.nanmean([auc(l, s) for l, s in zip(p_lb, p_sc)])),
            'micro_mm': auc(yp, np.concatenate([minmax(s) for s in p_sc])), 'n_video': len(p_te), 'n_frame': int(len(yp))}
    np.savez(f'{CFG.OUT_DIR}/skorlar/ped2_frame_scores.npz', *p_sc)
    print(f"Ped2 mikro AUC %{100*PED2['micro']:.2f} | makro %{100*PED2['macro']:.2f} | min-max %{100*PED2['micro_mm']:.2f}")
else:
    print('model_ped2.pth yok → Ped2 atlandı (01 notebook\'unu RUN=\'ped2\' ile çalıştırın)')

In [ ]:
# Literatür karşılaştırması (yalnızca kare düzeyi mikro AUC; değerleri orijinal makalelerden DOĞRULAYIN)
LIT = [('Conv-AE (Hasan ve ark., 2016)', 70.2, 90.0, 'Özgün makale değerleri'),
       ('Conv-AE (Liu ve ark. 2018 yeniden uygulaması)', 80.0, 85.0, 'Liu ve ark. 2018, Tablo'),
       ('ConvLSTM-AE (Luo ve ark., 2017)', 77.0, 88.1, ''),
       ('Gelecek kare tahmini (Liu ve ark., 2018)', 85.1, 95.4, ''),
       ('MemAE (Gong ve ark., 2019)', 83.3, 94.1, ''),
       ('MNAD-yeniden yapılandırma (Park ve ark., 2020)', 82.8, 90.2, ''),
       ('MNAD-tahmin (Park ve ark., 2020)', 88.5, 97.0, '')]
rows = [{'Yöntem': a, 'Avenue AUC %': b, 'Ped2 AUC %': c, 'Not': d} for a, b, c, d in LIT]
rows.append({'Yöntem': 'Önerilen 3B otokodlayıcı (bu çalışma)', 'Avenue AUC %': pct(R0[f'av_micro_s{AV_STRIDE}']),
             'Ped2 AUC %': pct(PED2['micro']) if PED2 else 'çalıştırılmadı', 'Not': 'Aynı protokol, kare düzeyi mikro AUC, ham skor'})
T['tablo1'] = pd.DataFrame(rows); display(T['tablo1'])

## F · K5 — Seed özeti, video sayıları, güven aralıkları

In [ ]:
keys = [('Avenue mikro AUC %', f'av_micro_s{AV_STRIDE}'), ('Avenue makro AUC %', f'av_macro_s{AV_STRIDE}'),
        ('UCF-Crime ROC-AUC %', 'cr_auc'), ('UCF-Crime PR-AUC %', 'cr_ap'),
        ('UCF-Crime yanlış alarm P95 %', 'cr_fpr_P95'), ('UCF-Crime recall P95 %', 'cr_rec_P95'),
        ('UCF-Crime yanlış alarm P99 %', 'cr_fpr_P99'), ('UCF-Crime recall P99 %', 'cr_rec_P99')]
if urf: keys += [('UR Fall ADL–düşme AUC % (K1-a)', 'urf_a'), ('UR Fall aynı video içi AUC % (K1-b)', 'urf_b'),
                 ('UR Fall yanlış alarm P95 %', 'urf_fpr_P95'), ('UR Fall recall P95 %', 'urf_rec_P95'),
                 ('UR Fall yanlış alarm P99 %', 'urf_fpr_P99'), ('UR Fall recall P99 %', 'urf_rec_P99')]
rows = []
for lab, k in keys:
    vals = [RES[t][k] for t in RES]
    r = {'Metrik': lab, **{t: pct(RES[t][k]) for t in RES}}
    r['Ortalama ± std'] = f'{100*np.mean(vals):.2f} ± {100*np.std(vals):.2f}' if len(vals) > 1 else '—'
    rows.append(r)
T['seed'] = pd.DataFrame(rows); display(T['seed'])
print(f'{len(RES)} model değerlendirildi:', list(RES))

In [ ]:
# Video kimliği: dosya adının sonundaki kare numarası atılarak bulunur (örn. Fighting003_x264_120.jpg → Fighting003_x264)
def video_ids(names):
    stems = [re.sub(r'[\s_\-\.]*\d+$', '', os.path.splitext(n)[0]) for n in names]
    u = sorted(set(stems), key=natural_key)
    ok = (1 < len(u) < len(names) / 5) or (len(u) == 1 and re.search(r'\d', u[0]) is not None)
    return (stems, u) if ok else (None, u)

vid_rows, VID = [], {}
for k, v in cross.items():
    stems, u = video_ids(v['names'])
    ok = stems is not None and len(u) >= 1 and any(c.isalpha() for c in u[0])
    VID[k] = [stems[w[0]] for w in v['idx']] if ok else None
    vid_rows.append({'Kategori': k, 'Kaynak': 'UCF-Crime',
                     'Video sayısı': len(u) if ok else 'dosya adından çıkarılamadı — elle yazın',
                     'Kare': len(v['frames']), 'Sekans': len(v['idx']), 'Etiket': 'Anomali' if v['label'] else 'Normal',
                     'Örnek dosya adı': v['names'][0]})
if urf:
    for kind, lab in [('adl', 'UR Fall ADL (K1)'), ('fall', 'UR Fall düşme (K1)')]:
        vs = [v for v in urf.values() if v['kind'] == kind]
        vid_rows.append({'Kategori': lab, 'Kaynak': 'UR Fall', 'Video sayısı': len(vs), 'Kare': sum(len(v['frames']) for v in vs),
                         'Sekans': sum(len(v['idx']) for v in vs), 'Etiket': 'Normal' if kind == 'adl' else 'Kare etiketli', 'Örnek dosya adı': ''})
vid_rows.append({'Kategori': 'fall_test (önceki sürüm — ÇIKARILDI)', 'Kaynak': 'UR Fall', 'Video sayısı': 75, 'Kare': 1550, 'Sekans': 772,
                 'Etiket': '600 ADL karesi yanlışlıkla anomali etiketliydi', 'Örnek dosya adı': 'fall_adl-01-cam0-rgb_...'})
vid_rows.append({'Kategori': 'Avenue test', 'Kaynak': 'CUHK Avenue', 'Video sayısı': len(av_frames),
                 'Kare': sum(map(len, av_frames)), 'Sekans': '—', 'Etiket': 'Kare etiketli', 'Örnek dosya adı': ''})
T['veri'] = pd.DataFrame(vid_rows); display(T['veri'])

# Güven aralığı: video kimlikleri varsa video düzeyi, yoksa blok bootstrap
rng = np.random.default_rng(CFG.SEED)
def units(k):
    s = M['cross'][k]
    if all(VID[c] is not None for c in cross):
        ids = np.array(VID[k]); return [s[ids == u] for u in dict.fromkeys(ids)], 'video düzeyi'
    return [s[i:i+CFG.BOOT_BLOCK] for i in range(0, len(s), CFG.BOOT_BLOCK)], f'blok ({CFG.BOOT_BLOCK} sekans)'
nu, boot_kind = units('Normal'); au = [u for c in cats for u in units(c)[0]]
ba, bp_ = [], []
for _ in range(CFG.N_BOOT):
    nb = np.concatenate([nu[i] for i in rng.integers(0, len(nu), len(nu))])
    ab = np.concatenate([au[i] for i in rng.integers(0, len(au), len(au))])
    yy = np.r_[np.zeros(len(nb)), np.ones(len(ab))]; ss_ = np.r_[nb, ab]
    ba.append(roc_auc_score(yy, ss_)); bp_.append(average_precision_score(yy, ss_))
CI = {'tur': boot_kind, 'roc': (np.percentile(ba, 2.5), np.percentile(ba, 97.5)), 'pr': (np.percentile(bp_, 2.5), np.percentile(bp_, 97.5))}
# Avenue: video düzeyi bootstrap (21 video)
y_av = av_labels; s_av = M['avenue'][AV_STRIDE]; bav = []
for _ in range(CFG.N_BOOT):
    pick = rng.integers(0, len(y_av), len(y_av)); yy = np.concatenate([y_av[i] for i in pick])
    if 0 < yy.mean() < 1: bav.append(roc_auc_score(yy, np.concatenate([s_av[i] for i in pick])))
CI['avenue'] = (np.percentile(bav, 2.5), np.percentile(bav, 97.5))
anom = np.concatenate([M['cross'][c] for c in cats]); nn_ = M['cross']['Normal']; bal = []
for i in range(CFG.N_BALANCED):
    sub = np.random.default_rng(i).choice(anom, len(nn_), replace=False)
    bal.append(roc_auc_score(np.r_[np.zeros(len(nn_)), np.ones(len(sub))], np.r_[nn_, sub]))
CI['balanced'] = (float(np.mean(bal)), float(np.std(bal)))
if urf:
    # UR Fall K1-a: 70 video üzerinden video düzeyi bootstrap
    per = []
    for k, v in urf.items():
        s_ = M['urf'][k]; ok = v['valid']
        if v['kind'] == 'adl': per.append((np.zeros(int(ok.sum())), s_[ok]))
        else: ev = v['event'] & ok; per.append((np.ones(int(ev.sum())), s_[ev]))
    bu = []
    for _ in range(CFG.N_BOOT):
        pick = rng.integers(0, len(per), len(per)); yy = np.concatenate([per[i][0] for i in pick])
        if 0 < yy.mean() < 1: bu.append(roc_auc_score(yy, np.concatenate([per[i][1] for i in pick])))
    CI['urf'] = (np.percentile(bu, 2.5), np.percentile(bu, 97.5))
    print(f"UR Fall ADL–düşme AUC %95 GA (video bootstrap, {len(per)} video): [{100*CI['urf'][0]:.2f}, {100*CI['urf'][1]:.2f}]")
print(f"Avenue mikro AUC %95 GA (video bootstrap): [{100*CI['avenue'][0]:.2f}, {100*CI['avenue'][1]:.2f}]")
print(f"UCF-Crime ROC-AUC %95 GA ({boot_kind}): [{100*CI['roc'][0]:.2f}, {100*CI['roc'][1]:.2f}] | PR-AUC: [{100*CI['pr'][0]:.2f}, {100*CI['pr'][1]:.2f}]")
print(f"Dengeli alt küme ({CFG.N_BALANCED} çekiliş): {100*CI['balanced'][0]:.2f} ± {100*CI['balanced'][1]:.2f}")

## G · K4 — Makale şekilleri (hepsi aynı skorlardan)

In [ ]:
FIG = f'{CFG.OUT_DIR}/sekiller'
thr0 = R0['thr']; tm = thr0[CFG.MAIN_THR]
plt.rcParams.update({'font.size': 10})

# Şekil A · Avenue skor eğrileri (4 video)
sc_av = M['avenue'][AV_STRIDE]
show = [v for v in range(len(av_frames)) if 0 < av_labels[v].mean() < 1][:4]
fig, axes = plt.subplots(len(show), 1, figsize=(10, 2.1*len(show)), squeeze=False)
for ax, v in zip(axes[:, 0], show):
    ax.plot(sc_av[v], lw=1.2, color='#1f5fa8', label='Anomali skoru (MSE)')
    ax.fill_between(range(len(av_labels[v])), 0, av_labels[v] * sc_av[v].max() * 1.05, color='#d1495b', alpha=0.2, step='mid', label='Gerçek anomali')
    ax.axhline(tm, color='k', ls='--', lw=1, label=f'Eşik ({CFG.MAIN_THR})')
    ax.set_ylabel(f'Video {v+1}'); ax.grid(alpha=0.3)
axes[0, 0].legend(loc='upper right', fontsize=8); axes[-1, 0].set_xlabel('Kare')
plt.tight_layout(); plt.savefig(f'{FIG}/sekil_avenue_skor.png', dpi=300); plt.show()

# Şekil B · Skor dağılımları + ROC (çapraz test)
ys = np.concatenate([np.full(len(M['cross'][k]), cross[k]['label']) for k in cross]); ss = np.concatenate([M['cross'][k] for k in cross])
fpr, tpr, _ = roc_curve(ys, ss)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bins = np.linspace(0, np.percentile(ss, 99.5), 70)
axes[0].hist(M['val'], bins=bins, alpha=.55, color='#7a7a7a', label='Avenue doğrulama (normal)')
axes[0].hist(nn_, bins=bins, alpha=.55, color='#1f5fa8', label='UCF-Crime normal')
axes[0].hist(anom, bins=bins, alpha=.45, color='#d1495b', label='Anomali (tüm kategoriler)')
for k, ls in [('P95', '--'), ('P99', ':')]: axes[0].axvline(thr0[k], color='k', ls=ls, label=f'Eşik {k}')
axes[0].set_xlabel('Yeniden yapılandırma hatası (MSE)'); axes[0].set_ylabel('Sekans sayısı'); axes[0].legend(fontsize=8); axes[0].grid(alpha=.3)
axes[1].plot(fpr, tpr, lw=2, color='#1f5fa8', label=f"Avenue→UCF-Crime, AUC = %{100*R0['cr_auc']:.2f}"); axes[1].plot([0,1],[0,1],'--',color='gray')
if urf:
    (ya_, sa2), _ = urf_sets(M['urf']); f_, t_, _ = roc_curve(ya_, sa2)
    axes[1].plot(f_, t_, lw=2, color='#d1495b', label=f"Avenue→UR Fall (ADL–düşme), AUC = %{100*R0['urf_a']:.2f}")
for k, m in [('P95','o'), ('P99','s')]:
    axes[1].scatter(R0[f'cr_fpr_{k}'], R0[f'cr_rec_{k}'], marker=m, s=60, color='k', zorder=5, label=f'Eşik {k} (UCF-Crime)')
axes[1].set_xlabel('Yanlış pozitif oranı'); axes[1].set_ylabel('Doğru pozitif oranı'); axes[1].legend(fontsize=8); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{FIG}/sekil_capraz_dagilim_roc.png', dpi=300); plt.show()

# Şekil C · Karışıklık matrisi (ana eşik)
cm = confusion_matrix(ys, (ss >= tm).astype(int), labels=[0, 1])
fig, ax = plt.subplots(figsize=(3.8, 3.4)); ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2): ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=13)
ax.set_xticks([0, 1], ['Normal', 'Anomali']); ax.set_yticks([0, 1], ['Normal', 'Anomali'])
ax.set_xlabel('Tahmin'); ax.set_ylabel('Gerçek'); ax.set_title(f'Eşik {CFG.MAIN_THR} = {tm:.1e}')
plt.tight_layout(); plt.savefig(f'{FIG}/sekil_karisiklik.png', dpi=300); plt.show()

# Şekil D · Kategori bazlı tespit oranı (eski "Klasör Bazlı" şeklin yerine)
fig, ax = plt.subplots(figsize=(9, 3.6)); xs = np.arange(len(cats) + 1 + (2 if urf else 0)); w = .38
for o, k, col in [(-w/2, 'P95', '#1f5fa8'), (w/2, 'P99', '#9bb7d9')]:
    vals = [100 * R0[f'cr_fpr_{k}']] + [100 * R0[f'cr_det_{c}_{k}'] for c in cats] + \
           ([100 * R0[f'urf_fpr_{k}'], 100 * R0[f'urf_rec_{k}']] if urf else [])
    b = ax.bar(xs + o, vals, w, color=col, label=f'Eşik {k}')
    ax.bar_label(b, fmt='%.1f', fontsize=7)
ax.set_xticks(xs, ['UCF normal\n(yanlış alarm)'] + [c.replace(' (', '\n(') for c in cats] + (['UR Fall ADL\n(yanlış alarm)', 'UR Fall\ndüşme anı'] if urf else []), fontsize=8)
ax.set_ylabel('Anomali kararı verilen sekans (%)')
ax.set_ylim(0, 110); ax.legend(); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.savefig(f'{FIG}/sekil_kategori_tespit.png', dpi=300); plt.show()

In [ ]:
# Şekil E · Hata haritaları — başlıktaki sayılar şekille aynı hesaptan gelir (K4: Şekil 5 tutarsızlığı)
mm = load_model(CFG.MODELS[CFG.MAIN]); samples = []
v = show[-1]; fr = av_frames[v]; ev = np.where(av_labels[v] == 1)[0]
def win_at(c, stride, n): s = int(np.clip(c - (CFG.SEQ_LEN//2) * stride, 0, n - (CFG.SEQ_LEN-1)*stride - 1)); return [s + t*stride for t in range(CFG.SEQ_LEN)]
assert win_at(0, AV_STRIDE, len(fr)) != win_at(int(ev[0]), AV_STRIDE, len(fr)) or len(fr) < 40
samples.append((f'Avenue video {v+1} — normal kare', fr, win_at(int(np.argmin(sc_av[v][:max(1, ev[0])])), AV_STRIDE, len(fr))))
samples.append((f'Avenue video {v+1} — anomali karesi', fr, win_at(int(ev[np.argmax(sc_av[v][ev])]), AV_STRIDE, len(fr))))
for c in ['Normal'] + cats:
    s = M['cross'][c]; j = int(np.argsort(s)[len(s)//2])   # medyan skorlu sekans (temsilî örnek)
    samples.append((f'{c} (medyan skorlu sekans)', cross[c]['frames'], cross[c]['idx'][j]))
caps = []
fig, axes = plt.subplots(len(samples), 3, figsize=(7.5, 2.3*len(samples)))
for r, (title, F, w) in enumerate(samples):
    s, rec = score_windows(mm, F, [w], return_recon=True); x = F[w]; mid = CFG.SEQ_LEN // 2
    err = np.abs(x[mid] - rec[0, mid])
    axes[r, 0].imshow(x[mid], cmap='gray', vmin=0, vmax=1); axes[r, 1].imshow(rec[0, mid], cmap='gray', vmin=0, vmax=1)
    im = axes[r, 2].imshow(err, cmap='hot', vmin=0, vmax=max(err.max(), 1e-6)); plt.colorbar(im, ax=axes[r, 2], fraction=.046)
    axes[r, 0].set_ylabel(title, fontsize=7); axes[r, 2].set_title(f'MSE={s[0]:.2e} ({s[0]/tm:.2f}× eşik)', fontsize=8)
    for a in axes[r]: a.set_xticks([]); a.set_yticks([])
    caps.append(f'{title}: sekans MSE = {s[0]:.2e}, eşiğin {s[0]/tm:.2f} katı')
axes[0, 0].set_title('Giriş (orta kare)', fontsize=8); axes[0, 1].set_title('Yeniden yapılandırma', fontsize=8)
plt.tight_layout(); plt.savefig(f'{FIG}/sekil_hata_haritalari.png', dpi=300); plt.show()
open(f'{FIG}/sekil_hata_haritalari_altyazi.txt', 'w', encoding='utf-8').write('\n'.join(caps))
print('\n'.join(caps))

# Şekil F · K1: model ve öğrenmesiz skorlar
k1 = T['k1'].set_index('Yöntem'); fig, ax = plt.subplots(figsize=(10, 4.8))
k1.plot.bar(ax=ax, width=.8, colormap='tab10'); ax.axhline(50, color='gray', ls=':', lw=1)
ax.set_ylabel('AUC (%)'); ax.set_xlabel(''); ax.set_ylim(0, 105); ax.grid(axis='y', alpha=.3)
ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.32), ncol=2, frameon=False)
plt.xticks(rotation=20, ha='right', fontsize=8); plt.tight_layout(); plt.savefig(f'{FIG}/sekil_k1_kontrol.png', dpi=300); plt.show()

# Şekil G · Seed eğitim eğrileri (01 notebook kayıtları)
logs = sorted(glob.glob(f'{CFG.LOG_DIR}/*.json'))
if logs:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for lp in logs:
        L = json.load(open(lp)); ax.plot(range(1, len(L['val'])+1), L['val'], label=f"{L['run']} (en iyi epoch {L['best_epoch']})")
    ax.set_xlabel('Epoch'); ax.set_ylabel('Doğrulama kaybı'); ax.legend(fontsize=8); ax.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(f'{FIG}/sekil_egitim_seedler.png', dpi=300); plt.show()

## Kaydet + makaleye girecek sayılar

In [ ]:
TD = f'{CFG.OUT_DIR}/tablolar'
for k, t in T.items(): t.to_csv(f'{TD}/{k}.csv', index=False, encoding='utf-8-sig')
with pd.ExcelWriter(f'{TD}/tum_tablolar.xlsx') as xw:
    for k, t in T.items(): t.to_excel(xw, sheet_name=k[:31], index=False)

f2 = lambda x: f'{100*x:.2f}'.replace('.', ',')
def _md(df):
    try: return df.to_markdown()
    except Exception: return df.to_string()
k1d = T['k1'].set_index('Yöntem')
lines = [
 '# Makaleye girecek sayılar (otomatik üretildi — elle değiştirmeyin, notebook\'u yeniden çalıştırın)', '',
 '## Protokol',
 f'- Eşik: yalnızca Avenue doğrulama kümesindeki {len(val_idx)} normal sekanstan; ana eşik {CFG.MAIN_THR} = {tm:.2e} (P99 = {thr0["P99"]:.2e}).',
 f'- Avenue test pencereleri: 8 kare, pencere içi kare aralığı {AV_STRIDE} ({stride_note}); kare skoru = onu içeren pencerelerin ortalaması.',
 f'- Avenue→UCF-Crime: {len(nn_)} normal + {len(anom)} anomali sekansı (tamamı, alt örnekleme yok).',
 '- Önceki sürümdeki fall_test klasörü çıkarıldı (600 ADL karesi anomali etiketliydi). Düşme testi kare etiketli UR Fall ile yapıldı.',
 '- Kategori adları düzeltildi: ucf_robbery = Stealing (hırsızlık), ucf_accident = Burglary (soygun), kaza değil.', '',
 '## K2 — Avenue resmi test kümesi (21 video, kare düzeyi)',
 f'- Mikro AUC: %{f2(R0[f"av_micro_s{AV_STRIDE}"])} (%95 GA, video bootstrap: {f2(CI["avenue"][0])}–{f2(CI["avenue"][1])})',
 f'- Makro AUC: %{f2(R0[f"av_macro_s{AV_STRIDE}"])}; video bazında min-max ile mikro AUC: %{f2(R0[f"av_micro_mm_s{AV_STRIDE}"])}',
 f'- Ped2 mikro AUC: ' + (f'%{f2(PED2["micro"])} (makro %{f2(PED2["macro"])})' if PED2 else 'çalıştırılmadı'), '',
 '## Avenue → UCF-Crime',
 f'- ROC-AUC %{f2(R0["cr_auc"])} (%95 GA, {CI["tur"]}: {f2(CI["roc"][0])}–{f2(CI["roc"][1])}); PR-AUC %{f2(R0["cr_ap"])} (anomali oranı %{100*len(anom)/(len(anom)+len(nn_)):.0f} olduğu için yüksek)',
 f'- Dengeli alt küme ({CFG.N_BALANCED} çekiliş): %{f2(CI["balanced"][0])} ± {f2(CI["balanced"][1])}',
 *[f'- {c}: AUC %{f2(R0[f"cr_auc_{c}"])}, tespit P95 %{f2(R0[f"cr_det_{c}_P95"])}, P99 %{f2(R0[f"cr_det_{c}_P99"])}' for c in cats],
 f'- Normal yanlış alarm: P95 %{f2(R0["cr_fpr_P95"])}, P99 %{f2(R0["cr_fpr_P99"])}',
 f'- Karışıklık matrisi ({CFG.MAIN_THR}): gerçek normal → {cm[0,0]} doğru / {cm[0,1]} yanlış alarm; gerçek anomali → {cm[1,1]} tespit / {cm[1,0]} kaçırılan', '',
 *([ '## Avenue → UR Fall (kare etiketli, 40 ADL + 30 düşme videosu)',
     f'- ADL–düşme anı AUC (K1-a): %{f2(R0["urf_a"])} (%95 GA, video bootstrap: {f2(CI["urf"][0])}–{f2(CI["urf"][1])})',
     f'- Aynı düşme videosu içinde düşme öncesi–düşme anı AUC (K1-b): %{f2(R0["urf_b"])}',
     f'- P95: ADL yanlış alarm %{f2(R0["urf_fpr_P95"])}, düşme recall %{f2(R0["urf_rec_P95"])} | P99: yanlış alarm %{f2(R0["urf_fpr_P99"])}, recall %{f2(R0["urf_rec_P99"])}', ''] if urf else []),
 '## K1 — Kontrol deneyleri (AUC %)',
 _md(k1d), '',
 '## K5 — Seed özeti', T['seed'].to_string(index=False), '',
 '## Şekil E alt yazı değerleri', *caps]
open(f'{CFG.OUT_DIR}/makale_sayilari.md', 'w', encoding='utf-8').write('\n'.join(lines))
json.dump({'RES': RES, 'CI': {k: [float(x) for x in np.atleast_1d(v)] if not isinstance(v, str) else v for k, v in CI.items()},
           'PED2': PED2, 'AV_STRIDE': AV_STRIDE, 'stride_note': stride_note},
          open(f'{CFG.OUT_DIR}/sonuclar_tam.json', 'w', encoding='utf-8'), indent=1, ensure_ascii=False, default=float)
print(open(f'{CFG.OUT_DIR}/makale_sayilari.md', encoding='utf-8').read())
print('\n✅ Her şey kaydedildi →', CFG.OUT_DIR)